# Predicting the Winner of the 2026 FIFA World Cup

Group Members: Mark Vincent Baclagan, Eli Bryan, Maryam Sayeb, Jasmine Sidhu

https://www.kaggle.com/datasets/cashncarry/fifaworldranking/data
\
https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017/data

## Introduction

With the 2026 FIFA World Cup now just a few months away, anticipation for the tournament is growing rapidly worldwide. In British Columbia, competition feels even more exciting for residents this year, as Vancouver is one of the official host cities and will host seven matches at BC Place. As the world's biggest sporting event approaches, predicting which of the 48 nations will win the tournament becomes a compelling data science question. By using historical World Cup results, international match data, and team rankings, it is possible to identify patterns that may help determine the strongest contenders for the 2026 title.

### Maybe like a few photos here?

The relevance of the World Cup in relation to Vancouver has motivated us to develop a prdeictive model to tend to the question; **Are we able to predict the nations with the largest chances to win the 2026 Word Cup with previous matches, and historical Word Cup results? Moreover, if we are able to make these predictions, what model would allow us to make this prediction?**

With our two data sets, we have variables related to different countries FIFA rankings, as well as previous match history.

**The first dataset(https://www.kaggle.com/datasets/cashncarry/fifaworldranking/data) consists of the following:**

* **country_full - Countries Full Name**

* **country_abrv - Countries Abbreviated Name**

* **rank - Current Country Rank**

* **total_points — Current Total Points**

* **previous_points — Total Points in Last Rating**

* **rank_change — How Rank Has Changed Since The Last Publication**

* **confederation — FIFA Confederations**

* **rank_date — Date of Rating Calculation**

**The second dataset(https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017/data) consists of the follwing columns:**

* **date - Date of The Match**

* **away_team - Name of The Away Team**

* **home_score - Full-time Home Team Score Including Extra Time (Not Including Penalty-Shootouts)**

* **away_score - Full-Time Away Team Score Including Extra Time (Not Including Penalty-Shootouts)**

* **tournament - Name of The Tournament**

* **city - Name of The City/Town/Administrative Unit Where The Match Was Played**

* **country - Name of The Cuntry The Match Was Played In**

* **neutral - TRUE/FALSE Column Indicating Whether The Match Was Played At a Neutral Venue**

### Methods & Results
To conduct our analysis, we chose to use a K-Nearest Neighbors classification model to predict match outcomes between international football teams.  We chose a classification model because our target — whether the home team wins or loses — is a binary outcome, making classification a natural fit.
To build this model, we planned and executed the following steps:

**Reading & Wrangling:** Loaded two datasets containing historical international match results and FIFA world rankings, then merged them to attach each team's ranking to their respective matches.

**Labeling Outcomes:** Created a home win column where a value of 1 indicates the home team won and 0 indicates a loss, then removed all drawn matches since they have no winner.

**Feature Engineering:** Added a ranking difference column to capture the strength gap between the home and away team, then dropped any rows with missing values in key columns.

**Train/Test Split:** Divided the data so the model trains on 75% of matches and is tested on the remaining 25% it has never seen, making sure both sets have a similar proportion of wins and losses.

**Preprocessing:** Standardized the numeric ranking columns so they are on the same scale, while leaving the neutral venue column unchanged since it is already a simple yes/no value.

**K-Nearest Neighbor Classification:** Tested different values of K to find which of the following gives the most accurate predictions, then used the best K to make final predictions on the test set.


### Reading in Relevant Libaries

In [ ]:
import altair as alt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### Loading The Results Dataset

In [ ]:
results_data = pd.read_csv('https://raw.githubusercontent.com/sayebm/Group_Project/refs/heads/main/results.csv')
results_data

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False
...,...,...,...,...,...,...,...,...,...
49066,2026-01-18,Bolivia,Panama,1,1,Friendly,Tarija,Bolivia,False
49067,2026-01-18,Grenada,Jamaica,0,1,Friendly,St. George's,Grenada,False
49068,2026-01-22,Panama,Mexico,0,1,Friendly,Panama City,Panama,False
49069,2026-01-25,Bolivia,Mexico,0,1,Friendly,Santa Cruz,Bolivia,False


### Sorting the Results Data

Sorting the data to only consider obervations(Matches) from 2015 onwards

In [ ]:
# Only considering matches starting after 2015
results_data['date'] = pd.to_datetime(results_data['date'])
results_filtered = results_data[results_data['date'] >= '2015-01-01']
results_filtered

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
38380,2015-01-04,Bahrain,Jordan,1,0,Friendly,Ballarat,Australia,True
38381,2015-01-04,Iran,Iraq,1,0,Friendly,Wollongong,Australia,True
38382,2015-01-04,South Korea,Saudi Arabia,2,0,Friendly,Parramatta,Australia,True
38383,2015-01-04,South Africa,Zambia,1,0,Friendly,Johannesburg,South Africa,False
38384,2015-01-05,China PR,Oman,4,1,Friendly,Penrith,Australia,True
...,...,...,...,...,...,...,...,...,...
49066,2026-01-18,Bolivia,Panama,1,1,Friendly,Tarija,Bolivia,False
49067,2026-01-18,Grenada,Jamaica,0,1,Friendly,St. George's,Grenada,False
49068,2026-01-22,Panama,Mexico,0,1,Friendly,Panama City,Panama,False
49069,2026-01-25,Bolivia,Mexico,0,1,Friendly,Santa Cruz,Bolivia,False


### Loading The Rankings Dataset

Loading the data and converting the rank_date variable into a 'date' For Python.

In [ ]:
rankings_data = pd.read_csv('https://raw.githubusercontent.com/sayebm/Group_Project/refs/heads/main/fifa_ranking-2024-06-20.csv')
rankings_data["year"] = pd.to_datetime(rankings_data["rank_date"]).dt.year
rankings_data


,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date,year
0,140.0,Brunei Darussalam,BRU,2.00,0.00,140,AFC,1992-12-31,1992
1,33.0,Portugal,POR,38.00,0.00,33,UEFA,1992-12-31,1992
2,32.0,Zambia,ZAM,38.00,0.00,32,CAF,1992-12-31,1992
3,31.0,Greece,GRE,38.00,0.00,31,UEFA,1992-12-31,1992
4,30.0,Algeria,ALG,39.00,0.00,30,CAF,1992-12-31,1992
...,...,...,...,...,...,...,...,...,...
67467,137.0,Kuwait,KUW,1098.42,1085.46,-2,AFC,2024-06-20,2024
67468,136.0,Lithuania,LTU,1100.66,1095.23,-1,UEFA,2024-06-20,2024
67469,135.0,Malaysia,MAS,1107.58,1094.54,-3,AFC,2024-06-20,2024
67470,133.0,Solomon Islands,SOL,1111.02,1111.02,1,OFC,2024-06-20,2024


### Filter The Rankings Data to Only Consider Rankings From 2015 Onwards and For Each Country to Only Have 1 Ranking per year

In [ ]:
# Only considering rankings starting after 2015 and only have 1 observation per year per team
rankings_filtered = (
    rankings_data[rankings_data['year'] >= 2015]
    .groupby(['country_full', 'year'])
    .last()
    .reset_index()
)
rankings_filtered

,country_full,year,rank,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,Afghanistan,2015,150.0,AFG,194.00,168.00,-6,AFC,2015-12-03
1,Afghanistan,2016,146.0,AFG,189.00,189.00,-1,AFC,2016-12-22
2,Afghanistan,2017,148.0,AFG,181.00,181.00,1,AFC,2017-12-21
3,Afghanistan,2018,147.0,AFG,1068.00,1068.00,0,AFC,2018-12-20
4,Afghanistan,2019,149.0,AFG,1052.00,1052.00,0,AFC,2019-12-19
...,...,...,...,...,...,...,...,...,...
2101,Zimbabwe,2020,108.0,ZIM,1181.00,1181.00,0,CAF,2020-12-10
2102,Zimbabwe,2021,121.0,ZIM,1138.44,1138.44,0,CAF,2021-12-23
2103,Zimbabwe,2022,125.0,ZIM,1138.56,1138.56,0,CAF,2022-12-22
2104,Zimbabwe,2023,124.0,ZIM,1144.56,1144.56,0,CAF,2023-12-21


### Matching Country Names for Data Sets

A few countries are spelled differently in the datasets, so we needed to make them consistent across both.

In [ ]:
name_mapping = {
    'USA':            'United States',
    'Korea Republic': 'South Korea',
    'Congo DR':       'DR Congo',
    "Cote d'Ivoire":  'Ivory Coast',
    'IR Iran':        'Iran',
}
rankings_data['country_full'] = rankings_data['country_full'].replace(name_mapping)
rankings_filtered['country_full'] = rankings_filtered['country_full'].replace(name_mapping)

### Creating a Variable to Hold Countires that Have Already Qualified for the 2026 World Cup

In [ ]:
world_cup_2026_teams = [
    # Host nations (auto-qualified)
    "United States", "Canada", "Mexico",
    # UEFA
    "Germany", "Portugal", "Spain", "France", "England", "Netherlands",
    "Belgium", "Austria", "Turkey", "Poland", "Serbia",
    # CONMEBOL
    "Argentina", "Colombia", "Uruguay", "Brazil", "Ecuador", "Paraguay",
    # CONCACAF
    "Panama", "Costa Rica", "Honduras", "Jamaica",
    # CAF
    "Morocco", "Egypt", "Senegal", "South Africa", "DR Congo",
    "Ivory Coast", "Nigeria", "Algeria", "Tunisia",
    # AFC
    "Iran", "South Korea", "Japan", "Saudi Arabia", "Australia",
    "Uzbekistan", "Jordan", "Iraq",
    # OFC
    "New Zealand",
]

### Filtered to Only Include Matches Played Between Two Teams That Have Qualified For The 2026 World Cup

In [ ]:
# Only includes teams currently qualified for the wolrd cup
results_wc2026 = results_filtered[
    results_filtered['home_team'].isin(world_cup_2026_teams) &
    results_filtered['away_team'].isin(world_cup_2026_teams)
]
results_wc2026

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
38381,2015-01-04,Iran,Iraq,1,0,Friendly,Wollongong,Australia,True
38382,2015-01-04,South Korea,Saudi Arabia,2,0,Friendly,Parramatta,Australia,True
38395,2015-01-11,Nigeria,Ivory Coast,0,1,Friendly,Abu Dhabi,United Arab Emirates,True
38396,2015-01-11,Tunisia,Algeria,1,1,Friendly,Radès,Tunisia,False
38399,2015-01-12,Jordan,Iraq,0,1,AFC Asian Cup,Brisbane,Australia,True
...,...,...,...,...,...,...,...,...,...
49062,2026-01-14,Senegal,Egypt,1,0,African Cup of Nations,Tangier,Morocco,True
49063,2026-01-14,Morocco,Nigeria,0,0,African Cup of Nations,Rabat,Morocco,False
49064,2026-01-17,Egypt,Nigeria,0,0,African Cup of Nations,Casablanca,Morocco,True
49065,2026-01-18,Morocco,Senegal,0,1,African Cup of Nations,Rabat,Morocco,False


### Filter to Only Include Rankings For Teams That Are Qualified For The 2026 World Cup

In [ ]:
# Only includes teams currently qualified for the wolrd cup
rankings_wc2026 = rankings_filtered[
    rankings_filtered['country_full'].isin(world_cup_2026_teams)
].sort_values(by=['rank_date', 'rank'], ascending=[False, True])
rankings_wc2026

,country_full,year,rank,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
89,Argentina,2024,1.0,ARG,1860.14,1858.00,0,CONMEBOL,2024-06-20
717,France,2024,2.0,FRA,1837.47,1840.59,0,UEFA,2024-06-20
199,Belgium,2024,3.0,BEL,1797.98,1795.23,0,UEFA,2024-06-20
279,Brazil,2024,4.0,BRA,1791.85,1788.65,-1,CONMEBOL,2024-06-20
627,England,2024,5.0,ENG,1787.88,1794.90,1,UEFA,2024-06-20
...,...,...,...,...,...,...,...,...,...
967,Jordan,2015,87.0,JOR,399.00,411.00,5,AFC,2015-12-03
360,Canada,2015,88.0,CAN,388.00,335.00,-14,CONCACAF,2015-12-03
917,Iraq,2015,89.0,IRQ,381.00,392.00,2,AFC,2015-12-03
847,Honduras,2015,101.0,HON,338.00,359.00,6,CONCACAF,2015-12-03


### Adding Each Nations FIFA Ranking For Each Match Played

In [ ]:
# So the 2 datasets can correlate the countries rank at the time of a match with each other

results_wc2026['year'] = results_wc2026['date'].dt.year
rankings_lookup = rankings_wc2026[['country_full', 'year', 'rank']]


results_wc2026 = (
    results_wc2026.merge(
    rankings_lookup.rename(columns={'country_full': 'home_team', 'rank': 'home_ranking'})
    ).merge(
    rankings_lookup.rename(columns={'country_full': 'away_team', 'rank': 'away_ranking'})
    )
)
results_wc2026 = results_wc2026[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'neutral', 'home_ranking', 'away_ranking']]
results_wc2026


/tmp/ipykernel_9089/3029627807.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_wc2026['year'] = results_wc2026['date'].dt.year


,date,home_team,away_team,home_score,away_score,tournament,neutral,home_ranking,away_ranking
0,2015-01-04,Iran,Iraq,1,0,Friendly,True,45.0,89.0
1,2015-01-04,South Korea,Saudi Arabia,2,0,Friendly,True,51.0,80.0
2,2015-01-11,Tunisia,Algeria,1,1,Friendly,False,40.0,28.0
3,2015-01-12,Jordan,Iraq,0,1,AFC Asian Cup,True,87.0,89.0
4,2015-01-16,Iraq,Japan,0,1,AFC Asian Cup,True,89.0,53.0
...,...,...,...,...,...,...,...,...,...
1018,2024-11-18,United States,Jamaica,4,2,CONCACAF Nations League,False,11.0,53.0
1019,2024-11-19,Mexico,Honduras,4,0,CONCACAF Nations League,False,15.0,78.0
1020,2024-11-19,Colombia,Ecuador,0,1,FIFA World Cup qualification,False,12.0,30.0
1021,2024-11-19,Brazil,Uruguay,1,1,FIFA World Cup qualification,False,4.0,14.0


### Functions that sorts the 'Tournament' column in the DataFrame

In [ ]:
'''
def get_tournament_types(t = 'tournament'):

    #Returns the different types of values in the tournaments column

    return results_data[t].drop_duplicates().sort_values()

tournaments = get_tournament_types()

tournaments
'''

"\ndef get_tournament_types(t = 'tournament'):\n    \n    #Returns the different types of values in the tournaments column\n    \n    return results_data[t].drop_duplicates().sort_values()\n\ntournaments = get_tournament_types()\n\ntournaments\n"

### Displays how many times each unique tournament type is mentioned in the data under the tournaments column

In [ ]:
#results_wc2026['tournament'].value_counts()

### Assigns a numerical weight to each tournament type (1 = most important, and 0 = meaningless)

I will not include the 'tournament' that only occured 2-3 times in our data.

We can manipulate these values in case our data in the end does not seem reasonable

In [ ]:
'''
tournament_weights = {
    'FIFA World Cup': 1.00,
    'Copa América' : 0.85,
    'UEFA Euro' : 0.85,
    'AFC Asian Cup': 0.85,
    'African Cup of Nations': 0.85,
    'Gold Cup' : 0.80,
    'Confedertions Cup': 0.75, # Lower because it was discontinuied in 2019 and the final tournamnet was in 2017
    'CONCACAF Nations League': 0.75,
    'UEFA Nations League': 0.75,
    'Superclásico de las Américas': 0.65,
    'FIFA World Cup qualification': 0.65,
    'EAFF Championship': 0.65,
    'Arab Cup': 0.60,
    'UEFA Euro qaulification': 0.55,
    'Copa América qualification': 0.55,
    'WAFF Championship': 0.55,
    'FIFA Series': 0.55,
    'Gulf Cup': 0.55,
    'African Cup of Nations qualification': 0.50,
    'UNCAF Cup': 0.50,
    'COSAFA Cup': 0.50,
    'CAFA Nations Cup': 0.5,
    'Kirin Challenge Cup': 0.35,
    'Kirin Cup': 0.35,
    'Soccer Ashes': 0.30,
    'Friendly': 0.20,
}

results_wc2026_copy = results_wc2026.copy()                             #Copies filtered data
results_wc2026_copy['tournaments_weight'] = (                           #Creates a new column called 'tournament_weight'
    results_wc2026_copy['tournament'].map(tournament_weights).fillna(0.30) #Assigns 0.30 to any tournament that was not mentioned
)

results_wc2026_copy[['tournament', 'tournament_weight']                     #displays the new columns,removes duplicates,
    ].drop_duplicates().sort_values('tournament_weight', ascending = False) # and sorts them from highest to lowest weight
'''

"\ntournament_weights = {\n    'FIFA World Cup': 1.00,\n    'Copa América' : 0.85,\n    'UEFA Euro' : 0.85,\n    'AFC Asian Cup': 0.85,\n    'African Cup of Nations': 0.85,\n    'Gold Cup' : 0.80,\n    'Confedertions Cup': 0.75, # Lower because it was discontinuied in 2019 and the final tournamnet was in 2017\n    'CONCACAF Nations League': 0.75,\n    'UEFA Nations League': 0.75,\n    'Superclásico de las Américas': 0.65,\n    'FIFA World Cup qualification': 0.65,\n    'EAFF Championship': 0.65,\n    'Arab Cup': 0.60,\n    'UEFA Euro qaulification': 0.55,\n    'Copa América qualification': 0.55,\n    'WAFF Championship': 0.55,\n    'FIFA Series': 0.55,\n    'Gulf Cup': 0.55,\n    'African Cup of Nations qualification': 0.50,\n    'UNCAF Cup': 0.50,\n    'COSAFA Cup': 0.50,\n    'CAFA Nations Cup': 0.5,\n    'Kirin Challenge Cup': 0.35,\n    'Kirin Cup': 0.35,\n    'Soccer Ashes': 0.30,\n    'Friendly': 0.20,\n}\n\nresults_wc2026_copy = results_wc2026.copy()                             

### Labelling Winners and Cleaning the Data

We marked each match as a home win(1) or loss (0), removed draws since there is no winner. Also added a column that depicted the ranking gap between the two teams.

In [ ]:
results_wc2026["home_win"] = (results_wc2026["home_score"] > results_wc2026["away_score"]).astype(int)
results_wc2026["draw"] = results_wc2026["home_score"] == results_wc2026["away_score"]


model_data = results_wc2026.loc[~results_wc2026["draw"]].copy() #only rows that matches are not a draw


model_data["rank_diff"] = model_data["away_ranking"] - model_data["home_ranking"]#comapre strength


model_data = model_data.dropna(subset=["home_ranking", "away_ranking", "neutral"])
model_data

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_ranking,away_ranking,home_win,draw,rank_diff
0,2015-01-04,Iran,Iraq,1,0,Friendly,True,45.0,89.0,1,False,44.0
1,2015-01-04,South Korea,Saudi Arabia,2,0,Friendly,True,51.0,80.0,1,False,29.0
3,2015-01-12,Jordan,Iraq,0,1,AFC Asian Cup,True,87.0,89.0,0,False,2.0
4,2015-01-16,Iraq,Japan,0,1,AFC Asian Cup,True,89.0,53.0,0,False,-36.0
5,2015-01-17,Australia,South Korea,0,1,AFC Asian Cup,False,57.0,51.0,0,False,-6.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1016,2024-11-15,Uruguay,Colombia,3,2,FIFA World Cup qualification,False,14.0,12.0,1,False,-2.0
1018,2024-11-18,United States,Jamaica,4,2,CONCACAF Nations League,False,11.0,53.0,1,False,42.0
1019,2024-11-19,Mexico,Honduras,4,0,CONCACAF Nations League,False,15.0,78.0,1,False,63.0
1020,2024-11-19,Colombia,Ecuador,0,1,FIFA World Cup qualification,False,12.0,30.0,0,False,18.0


### Selecting our Features and Target
Obtains the columns we want to train and what our target is.

X holds the columns the model will learn (similar rankings and raking gap)
\
y holds the answer we want to predict - whether the home team wins or not

In [ ]:
X = model_data [[
    "home_ranking",
    "away_ranking",
    "rank_diff",
    "neutral",
]]

y= model_data["home_win"]
X

,home_ranking,away_ranking,rank_diff,neutral
0,45.0,89.0,44.0,True
1,51.0,80.0,29.0,True
3,87.0,89.0,2.0,True
4,89.0,53.0,-36.0,True
5,57.0,51.0,-6.0,False
...,...,...,...,...
1016,14.0,12.0,-2.0,False
1018,11.0,53.0,42.0,False
1019,15.0,78.0,63.0,False
1020,12.0,30.0,18.0,False


### Split Into Training and Testing Set

We divided our data so that the model learns from 75% of the matches. We then followed up by testing how well it learned on the remiaining 25% it did not review yet.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=123,
    stratify=y,
)

### Scale Our Numeric Features for the Model

We standardized the ranking columns so they are on the same scale (helping the model compare them fairly), while leaving the neutral venue column untouched since it is already as imple yes/no value.

In [ ]:
numeric_features = ["home_ranking", "away_ranking", "rank_diff"]
binary_features = ["neutral"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        ("bin", "passthrough", binary_features),
    ],
    remainder="drop",
)

### Finding the Best K Value for our KNN Model

We test different values of K (how many neighboring matches the model looks at) to find the specific one that gives us the most accurate predictions.

In [26]:
knn_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier()),
])

param_grid = {
    "model__n_neighbors": range(1, 21, 2),
}

knn_grid = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)

knn_grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['home_ranking',
                                                                          'away_ranking',
                                                                          'rank_diff']),
                                                                        ('bin',
                                                                         'passthrough',
                                                                         ['neutral'])])),
                                       ('model', KNeighborsClassifier())]),
             n_jobs=-1, param_grid={'model__n_neighbors': range(1, 21, 2)},
             scoring='accuracy')

### Model Accuracy and Performance

We evaluated our trained KNN model on the test set, reporting the best K value, overall accuracy, precision and recall.

In [27]:
best_k = knn_grid.best_params_['model__n_neighbors']
y_pred = knn_grid.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

print(f"Best K: {best_k}")
print(f"Test Accuracy: {test_accuracy:.2%}")
print()
print(classification_report(y_test, y_pred, target_names=["Away Win", "Home Win"]))

Best K: 11
Test Accuracy: 65.96%

              precision    recall  f1-score   support

    Away Win       0.55      0.41      0.47        69
    Home Win       0.70      0.81      0.75       119

    accuracy                           0.66       188
   macro avg       0.62      0.61      0.61       188
weighted avg       0.65      0.66      0.65       188



### Cross-Validation Accuracy by K

We visualize the cross-validation accuracy for each value of K tested to understand why K=11 was selected.

In [32]:
import altair as alt
import pandas as pd

cv_results = pd.DataFrame(knn_grid.cv_results_)
k_accuracy_chart = alt.Chart(cv_results).mark_line(point=True, color='steelblue').encode(
    x=alt.X('param_model__n_neighbors:Q', title='Number of Neighbors (K)'),
    y=alt.Y('mean_test_score:Q', title='Cross-Validation Accuracy', scale=alt.Scale(zero=False)),
    tooltip=['param_model__n_neighbors', 'mean_test_score']
).properties(
    title='KNN Cross-Validation Accuracy by K',
    width=500,
    height=300
)
k_accuracy_chart

alt.Chart(...)

### Confusion Matrix

We display a confusion matrix to better understand where our model makes correct and incorrect predictions.

In [29]:
cm = confusion_matrix(y_test, y_pred)
pd.crosstab(y_test, y_pred)

col_0,0,1
home_win,,
0,28,41
1,23,96


### Example Predictions

We demonstrate our model's predictions on a few hypothetical 2026 World Cup matchups.

In [33]:
example_matches = pd.DataFrame({
    'home_team':    ['Argentina', 'France',  'Morocco'],
    'away_team':    ['Brazil',    'England', 'Canada'],
    'home_ranking': [1.0,         2.0,       14.0],
    'away_ranking': [4.0,         5.0,       47.0],
    'rank_diff':    [3.0,         3.0,       33.0],
    'neutral':      [True,        True,      True],
})

X_examples = example_matches[['home_ranking', 'away_ranking', 'rank_diff', 'neutral']]
example_matches['predicted_home_win'] = knn_grid.predict(X_examples)
example_matches['prediction'] = example_matches['predicted_home_win'].map({1: 'Home Win', 0: 'Away Win'})

example_matches[['home_team', 'away_team', 'home_ranking', 'away_ranking', 'prediction']]

,home_team,away_team,home_ranking,away_ranking,prediction
0,Argentina,Brazil,1.0,4.0,Home Win
1,France,England,2.0,5.0,Home Win
2,Morocco,Canada,14.0,47.0,Home Win


### Discussion & Conclusion

Our K-nearest neighbors classification model achieved a test accuracy of approximately 66% in predicting match outcomes between 2026 FIFA World Cup qualified nations. Using cross-validation, we determined that K=11 neighbors produced the best balance between overfitting and underfitting.

The results suggest that FIFA ranking-based features, particularly the ranking difference between teams, are moderately useful predictors of match outcomes. The model performed better at predicting home wins (precision: 70%, recall: 81%) than away wins (precision: 55%, recall: 41%), which is consistent with the well-documented home advantage phenomenon in football.

However, a 66% accuracy highlights the inherent difficulty of predicting football outcomes. Our model does not capture important factors such as player availability, recent form, or tournament context, and excluding draws from training simplifies the problem considerably. Future work could explore additional features or alternative classification methods to improve predictive performance.